# Exploratory Data Analysis (EDA)
## ML-Based Intelligent System for Automated Sleep Disorder Detection Using Lifestyle and Health Data

> **Academic Final Year Project**  
> **Dataset**: Sleep Health and Lifestyle Dataset (374 samples, 13 raw attributes)  
> **Target Classes**: `None` (Healthy baseline), `Insomnia`, `Sleep Apnea`  

> [!IMPORTANT]
> **Medical Disclaimer**: This research project is intended solely for academic inquiry and educational analysis. The outputs of statistical models and machine learning pipelines do not represent medical diagnoses. Associations documented here reflect analytical patterns within a synthetic dataset.

### 1. Environment Setup & Module Imports
We begin by importing required numerical, statistical, and plotting libraries. We also import the pre-tested, modular pipeline components from `ml.preprocess` and `ml.feature_engineering` to ensure end-to-end consistency across experiments.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for local module imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ml.preprocess import clean_and_prepare_data, TARGET_CLASSES, ID_COL, TARGET_COL
from ml.feature_engineering import FeatureEngineer

# Visualization styling
sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = {'None': '#2ca02c', 'Insomnia': '#ff7f0e', 'Sleep Apnea': '#d62728'}
CLASS_ORDER = ['None', 'Insomnia', 'Sleep Apnea']
print('Libraries and custom ML modules imported successfully.')

### 2. Dataset Loading and Schema Inspection
We load the raw CSV file directly from the `data/` directory and inspect the dimensionality, column data types, missing values, and duplicate instances.

In [ ]:
data_path = os.path.join(project_root, 'data', 'Sleep_health_and_lifestyle_dataset.csv')
raw_df = pd.read_csv(data_path)
print(f'Raw dataset shape: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns')
display(raw_df.head(5))
print('\nData Types and Null Counts:')
print(raw_df.info())

### 3. Data Integrity & Missing Values Check
We check for missing or NULL cells across all 13 attributes, as well as exact duplicates and feature-level duplicate profiles.

In [ ]:
missing_counts = raw_df.isnull().sum()
print('Missing Values per Column:')
print(missing_counts)
print(f'Total Missing Cells: {missing_counts.sum()}')
print(f'Exact Row Duplicates: {raw_df.duplicated().sum()}')
print(f'Duplicates excluding Person ID: {raw_df.drop(columns=[ID_COL]).duplicated().sum()}')

**Integrity Observation**:
- The dataset is 100% complete with 0 missing cells.
- `Person ID` is an artificial index and must be strictly excluded from ML inputs.
- While there are 0 exact duplicates, 45 rows share identical feature combinations, representing identical synthetic personas.

### 4. Data Preprocessing & Feature Engineering Pipeline Verification
We now invoke `clean_and_prepare_data()` from `ml.preprocess` and `FeatureEngineer()` from `ml.feature_engineering`.
This step drops `Person ID`, normalizes `BMI Category` (`Normal Weight` -> `Normal`), parses `Blood Pressure` into `BP_Systolic` and `BP_Diastolic`, preserves `'None'` as a valid target, and calculates clinical ratio features.

In [ ]:
X, y = clean_and_prepare_data(raw_df)
fe = FeatureEngineer(add_ratios=True)
X_eng = fe.fit_transform(X)

print(f'Preprocessed Feature Matrix Shape: {X_eng.shape}')
print(f'Engineered Features: {fe.engineered_feature_names_}')
display(X_eng.head(3))

### 5. Target Class Distribution & Class Imbalance Analysis
We analyze the distribution of the target variable `Sleep Disorder`. Notice that `'None'` is preserved as the healthy reference class without artificial oversampling during EDA.

In [ ]:
counts = y.value_counts()[CLASS_ORDER]
pcts = (counts / len(y)) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
bars = ax1.bar(CLASS_ORDER, counts.values, color=[PALETTE[c] for c in CLASS_ORDER], edgecolor='black', alpha=0.85)
ax1.set_title('Target Class Frequencies', fontweight='bold')
ax1.set_ylabel('Number of Patients')
for bar in bars:
    h = bar.get_height()
    ax1.annotate(f'{int(h)} ({h/len(y)*100:.1f}%)', (bar.get_x() + bar.get_width()/2, h), ha='center', va='bottom', fontweight='bold')

ax2.pie(counts.values, labels=CLASS_ORDER, autopct='%1.1f%%', colors=[PALETTE[c] for c in CLASS_ORDER], startangle=140, wedgeprops=dict(width=0.45, edgecolor='black'))
ax2.set_title('Class Proportions', fontweight='bold')
plt.tight_layout()
plt.show()

**Class Imbalance Insights**:
- Healthy baseline (`None`) accounts for 58.6% of the dataset.
- The two disorders are almost evenly split: `Insomnia` (20.6%) and `Sleep Apnea` (20.9%).
- Downstream models should use **Stratified 5-Fold Cross-Validation** and be evaluated using **Macro F1-Score** and **Balanced Accuracy**.

### 6. Descriptive Statistics (Parametric & Non-Parametric)
We calculate both parametric (Mean, Standard Deviation) and non-parametric (Median, IQR, Range, Skewness) metrics for continuous variables.

In [ ]:
numeric_cols = list(X_eng.select_dtypes(include=[np.number]).columns)
stats_df = pd.DataFrame({
    'Mean': X_eng[numeric_cols].mean(),
    'Std': X_eng[numeric_cols].std(),
    'Median': X_eng[numeric_cols].median(),
    'IQR': X_eng[numeric_cols].quantile(0.75) - X_eng[numeric_cols].quantile(0.25),
    'Min': X_eng[numeric_cols].min(),
    'Max': X_eng[numeric_cols].max(),
    'Skewness': X_eng[numeric_cols].skew(),
}).round(2)
display(stats_df)

### 7. Categorical Predictor Associations (Gender, Occupation, BMI Category)
We cross-tabulate each categorical feature with the target disorder to detect demographic patterns.

In [ ]:
merged = X_eng.copy()
merged['Sleep Disorder'] = y.values

for cat_col in ['Gender', 'BMI Category', 'Occupation']:
    print(f'=== Cross-Tabulation: {cat_col} vs Sleep Disorder (Percentages) ===')
    ct = pd.crosstab(merged[cat_col], merged['Sleep Disorder'], normalize='index')[CLASS_ORDER] * 100
    display(ct.round(1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=merged, x='Gender', hue='Sleep Disorder', hue_order=CLASS_ORDER, palette=PALETTE, ax=axes[0])
axes[0].set_title('Gender vs Disorder', fontweight='bold')
sns.countplot(data=merged, x='BMI Category', hue='Sleep Disorder', hue_order=CLASS_ORDER, palette=PALETTE, ax=axes[1])
axes[1].set_title('BMI Category vs Disorder', fontweight='bold')
sns.countplot(data=merged, y='Occupation', hue='Sleep Disorder', hue_order=CLASS_ORDER, palette=PALETTE, ax=axes[2])
axes[2].set_title('Occupation vs Disorder', fontweight='bold')
plt.tight_layout()
plt.show()

**Categorical Insights**:
- **BMI Category**: 90.7% of participants in the `Normal` BMI category have no disorder. Conversely, over 88% of `Overweight` and 90% of `Obese` participants suffer from either Insomnia or Sleep Apnea.
- **Occupation**: Nurses exhibit high concentration of Sleep Apnea, whereas Teachers and Salespersons exhibit elevated Insomnia.
- **Gender**: Males exhibit relatively higher Insomnia rates, while Females exhibit higher Sleep Apnea rates in this cohort.

### 8. Health & Lifestyle Metric Distributions by Sleep Disorder
We examine boxplot distributions of continuous variables across disorder groups to assess class separability.

In [ ]:
key_vars = ['Sleep Duration', 'Quality of Sleep', 'Stress Level', 'BP_Systolic', 'BP_Diastolic', 'Heart Rate']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for idx, var in enumerate(key_vars):
    sns.boxplot(data=merged, x='Sleep Disorder', y=var, order=CLASS_ORDER, palette=PALETTE, ax=axes[idx], showmeans=True)
    axes[idx].set_title(var, fontweight='bold')
plt.suptitle('Clinical & Lifestyle Variance Across Disorder Groups', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 9. Correlation Heatmaps & Multicollinearity Inspection
We compute Pearson and Spearman correlation matrices across baseline and engineered numerical attributes.

In [ ]:
plt.figure(figsize=(12, 9))
corr = X_eng[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix of All Health & Engineered Features', fontweight='bold', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

> [!WARNING]
> **Observational Correlation Caution**: Note that while Sleep Duration and Stress Level are strongly negatively correlated ($r = -0.81$), and Systolic Blood Pressure strongly associates with Sleep Apnea ($r = 0.58$), **correlation does not imply causation**. These patterns represent physiological signatures suitable for predictive classification, but not causal mechanisms.

### 10. Conclusion and Machine Learning Recommendations

In summary, the Exploratory Data Analysis reveals:
1. **Featurization**: Splitting `Blood Pressure` and normalizing `BMI Category` provides strong discriminative signals. The engineered features `BP_Pulse_Pressure` and `Sleep_to_Stress_Ratio` capture significant non-linear relationships.
2. **Algorithm Suitability**: Tree-based ensembles (`RandomForestClassifier`, `XGBClassifier`) are prime candidates given their natural handling of collinear features (`BP_Systolic` vs `BP_Diastolic`) and categorical variables.
3. **Validation Strategy**: Use 5-fold Stratified Cross-Validation with Macro F1-score as the lead selection metric to prevent majority class dominance.
4. **Ethical Safeguard**: All predictions in downstream phases must feature explicit non-diagnostic disclaimers.